In [2]:
# Install any missing libraries
import subprocess
subprocess.run(["pip", "install", "scikit-learn", "pandas", "boto3", "joblib"], capture_output=True)

CompletedProcess(args=['pip', 'install', 'scikit-learn', 'pandas', 'boto3', 'joblib'], returncode=0, stdout=b'Requirement already satisfied: scikit-learn in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (1.7.2)\nRequirement already satisfied: pandas in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (2.3.3)\nRequirement already satisfied: boto3 in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (1.42.84)\nRequirement already satisfied: joblib in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (1.5.3)\nRequirement already satisfied: numpy>=1.22.0 in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (from scikit-learn) (1.26.4)\nRequirement already satisfied: scipy>=1.8.0 in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (from scikit-learn) (1.17.1)\nRequirement already satisfied: threadpoolctl>=3.1.0 in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (fro

In [3]:
import boto3
import pandas as pd
from io import StringIO

# Download processed CSVs from S3
s3 = boto3.client('s3', region_name='us-east-2')
bucket = 'aws-ids-platform'
prefix = 'processed/friday-ddos-clean/'

# List all part files
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
files = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.csv')]

# Load and combine all parts
dfs = []
for f in files:
    obj = s3.get_object(Bucket=bucket, Key=f)
    dfs.append(pd.read_csv(obj['Body']))

df = pd.concat(dfs, ignore_index=True)
print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['Label'].value_counts()}")

Dataset shape: (225741, 79)
Label distribution:
Label
DDoS      128027
BENIGN     97714
Name: count, dtype: int64


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# Prepare features and labels
X = df.drop(columns=['Label'])
y = (df['Label'] == 'DDoS').astype(int)  # 1 = attack, 0 = normal

# Replace infinities with NaN then drop
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna(axis=1)  # drop columns with any NaN
y = y[X.index]        # keep labels aligned

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {X_train.shape[0]} samples with {X_train.shape[1]} features")

# Train model
model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

# Evaluate
print("\nClassification Report:")
print(classification_report(y_test, model.predict(X_test)))

Training on 180592 samples with 76 features

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19417
           1       1.00      1.00      1.00     25732

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149



In [5]:
import joblib
import os

# Save model locally in the notebook
joblib.dump(model, '/tmp/rf_model.joblib')
print("Model saved locally")

# Upload to S3
s3.upload_file('/tmp/rf_model.joblib', 'aws-ids-platform', 'models/rf_model.joblib')
print("Model uploaded to s3://aws-ids-platform/models/rf_model.joblib")

# Also save the feature column names — needed later for Lambda inference
import json
feature_cols = list(X.columns)
with open('/tmp/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

s3.upload_file('/tmp/feature_cols.json', 'aws-ids-platform', 'models/feature_cols.json')
print(f"Feature columns saved ({len(feature_cols)} features)")

Model saved locally
Model uploaded to s3://aws-ids-platform/models/rf_model.joblib
Feature columns saved (76 features)


In [6]:
import sklearn
print(sklearn.__version__)

1.7.2


In [7]:
import subprocess
subprocess.run(["pip", "install", "scikit-learn==1.7.2"], capture_output=True)

import importlib, sklearn
importlib.reload(sklearn)

# Retrain and resave with 1.7.2
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

import joblib
joblib.dump(model, '/tmp/rf_model.joblib')
s3.upload_file('/tmp/rf_model.joblib', 'aws-ids-platform', 'models/rf_model.joblib')
print("Model resaved with sklearn 1.7.2")

Model resaved with sklearn 1.7.2


In [8]:
# Get a real DDoS row to use as test
ddos_row = df[df['Label'] == 'DDoS'].iloc[0]
test_payload = ddos_row[feature_cols].to_dict()
test_payload['event_id'] = 'test-real-ddos-001'
test_payload['timestamp'] = '2026-05-01T02:00:00Z'

import json
print(json.dumps(test_payload))

{"Destination Port": 80.0, "Flow Duration": 1293792.0, "Total Fwd Packets": 3.0, "Total Backward Packets": 7.0, "Total Length of Fwd Packets": 26.0, "Total Length of Bwd Packets": 11607.0, "Fwd Packet Length Max": 20.0, "Fwd Packet Length Min": 0.0, "Fwd Packet Length Mean": 8.666667, "Fwd Packet Length Std": 10.263203, "Bwd Packet Length Max": 5840.0, "Bwd Packet Length Min": 0.0, "Bwd Packet Length Mean": 1658.1428, "Bwd Packet Length Std": 2137.297, "Flow IAT Mean": 143754.67, "Flow IAT Std": 430865.8, "Flow IAT Max": 1292730.0, "Flow IAT Min": 2.0, "Fwd IAT Total": 747.0, "Fwd IAT Mean": 373.5, "Fwd IAT Std": 523.9661, "Fwd IAT Max": 744.0, "Fwd IAT Min": 3.0, "Bwd IAT Total": 1293746.0, "Bwd IAT Mean": 215624.33, "Bwd IAT Std": 527671.94, "Bwd IAT Max": 1292730.0, "Bwd IAT Min": 2.0, "Fwd PSH Flags": 0.0, "Bwd PSH Flags": 0.0, "Fwd URG Flags": 0.0, "Bwd URG Flags": 0.0, "Fwd Header Length34": 72.0, "Bwd Header Length": 152.0, "Fwd Packets/s": 2.3187654, "Bwd Packets/s": 5.4104524,

In [9]:
import boto3
import pandas as pd
import numpy as np
from io import StringIO

s3 = boto3.client('s3', region_name='us-east-2')
bucket = 'aws-ids-platform'
prefix = 'processed/all-attacks-clean/'

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
files = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.csv')]

print(f"Loading sample from {len(files)} files...")
dfs = []
for i, f in enumerate(files):
    obj = s3.get_object(Bucket=bucket, Key=f)
    chunk = pd.read_csv(obj['Body'])
    if len(chunk) > 50000:
        chunk = chunk.sample(n=50000, random_state=42)
    dfs.append(chunk)
    print(f"  {i+1}/{len(files)}: {len(chunk)} rows")

df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal shape: {df.shape}")
print(f"\nLabel distribution:\n{df['Label'].value_counts()}")

Loading sample from 13 files...
  1/13: 50000 rows
  2/13: 50000 rows
  3/13: 50000 rows
  4/13: 50000 rows
  5/13: 50000 rows
  6/13: 50000 rows
  7/13: 50000 rows
  8/13: 50000 rows
  9/13: 50000 rows
  10/13: 50000 rows
  11/13: 50000 rows
  12/13: 50000 rows
  13/13: 50000 rows

Total shape: (650000, 79)

Label distribution:
Label
BENIGN          532289
DoS              57058
DDoS             28572
PortScan         27896
BruteForce        3485
Other              495
WebAttack          196
Infiltration         8
Heartbleed           1
Name: count, dtype: int64


In [ ]:
# Install imbalanced-learn
import subprocess
subprocess.run(["pip", "install", "imbalanced-learn"], capture_output=True)

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd
import numpy as np
import gc  # Garbage Collector

# Prepare features
X = df.drop(columns=['Label'])
y = df['Label']
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna(axis=1)
y = y[X.index]

# Downcast float64 to float32 (Cuts RAM usage by 50%)
for col in X.select_dtypes(include=['float64']).columns:
    X[col] = X[col].astype('float32')

print(f"Original counts:\n{y.value_counts()}\n")

TARGET_MAX = 30000
TARGET_MIN = 5000

class_counts = y.value_counts()

# Pre-process extreme minorities (Overwrite X and y directly)
print("Step 1: Over-sampling extreme minorities...")
pre_smote_strategy = {label: max(count, 6) for label, count in class_counts.items()}
ros = RandomOverSampler(sampling_strategy=pre_smote_strategy, random_state=42)
X, y = ros.fit_resample(X, y)

# Manually clear RAM
gc.collect()

# Undersample majority classes first (Massively reduces RAM needed for SMOTE)
print("Step 2: Under-sampling majorities...")
under_strategy = {label: min(count, TARGET_MAX) for label, count in y.value_counts().items()}
rus = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
X, y = rus.fit_resample(X, y)

gc.collect()

# Apply SMOTE to synthesize brand new attack boundaries
print("Step 3: Applying SMOTE...")
over_strategy = {label: max(under_strategy[label], TARGET_MIN) for label in under_strategy.keys()}
smote = SMOTE(sampling_strategy=over_strategy, random_state=42, k_neighbors=5)

# We output to X_bal and y_bal so that our downstream training code still works seamlessly
X_bal, y_bal = smote.fit_resample(X, y)

print(f"\nBalanced shape: {X_bal.shape}")
print(f"\nBalanced distribution:\n{y_bal.value_counts()}")

# Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal)

print(f"\nTraining on {X_train.shape[0]} samples with {X_train.shape[1]} features")

Original counts:
Label
BENIGN          532289
DoS              57058
DDoS             28572
PortScan         27896
BruteForce        3485
Other              495
WebAttack          196
Infiltration         8
Heartbleed           1
Name: count, dtype: int64

Step 1: Over-sampling extreme minorities...
Step 2: Under-sampling majorities...
Step 3: Applying SMOTE...

Balanced shape: (141468, 76)

Balanced distribution:
Label
BENIGN          30000
DoS             30000
DDoS            28572
PortScan        27896
BruteForce       5000
Heartbleed       5000
Infiltration     5000
Other            5000
WebAttack        5000
Name: count, dtype: int64

Training on 113174 samples with 76 features


In [11]:
# Train Random Forest multi-class classifier
model = RandomForestClassifier(
    n_estimators=100,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced'
)

print("Training model...")
model.fit(X_train, y_train)

print("\nEvaluating...")
y_pred = model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training model...

Evaluating...

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      6000
  BruteForce       0.99      0.95      0.97      1000
        DDoS       1.00      1.00      1.00      5715
         DoS       1.00      1.00      1.00      6000
  Heartbleed       1.00      1.00      1.00      1000
Infiltration       1.00      1.00      1.00      1000
       Other       0.99      0.99      0.99      1000
    PortScan       1.00      1.00      1.00      5579
   WebAttack       0.95      0.99      0.97      1000

    accuracy                           1.00     28294
   macro avg       0.99      0.99      0.99     28294
weighted avg       1.00      1.00      1.00     28294



In [12]:
import joblib
import json

# Save feature columns
feature_cols = list(X_bal.columns)

# Save model
joblib.dump(model, '/tmp/rf_model_multiclass.joblib')
s3.upload_file('/tmp/rf_model_multiclass.joblib', 'aws-ids-platform', 'models/rf_model.joblib')
print("Model uploaded")

# Save updated feature columns
with open('/tmp/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)
s3.upload_file('/tmp/feature_cols.json', 'aws-ids-platform', 'models/feature_cols.json')
print("Feature columns uploaded")

# Save label classes
classes = list(model.classes_)
with open('/tmp/classes.json', 'w') as f:
    json.dump(classes, f)
s3.upload_file('/tmp/classes.json', 'aws-ids-platform', 'models/classes.json')
print(f"Classes saved: {classes}")

Model uploaded
Feature columns uploaded
Classes saved: ['BENIGN', 'BruteForce', 'DDoS', 'DoS', 'Heartbleed', 'Infiltration', 'Other', 'PortScan', 'WebAttack']


In [13]:
attack_types = ['DDoS', 'DoS', 'PortScan', 'BruteForce', 'WebAttack']

for attack in attack_types:
    row = df[df['Label'] == attack].iloc[0]
    test_payload = row[feature_cols].to_dict()
    test_payload['event_id'] = f'test-{attack.lower()}-001'
    test_payload['timestamp'] = '2026-05-01T10:00:00Z'
    print(f"\n{attack}:")
    print(json.dumps(test_payload))


DDoS:
{"Destination Port": 80.0, "Flow Duration": 519166.0, "Total Fwd Packets": 3.0, "Total Backward Packets": 5.0, "Total Length of Fwd Packets": 26.0, "Total Length of Bwd Packets": 11601.0, "Fwd Packet Length Max": 20.0, "Fwd Packet Length Min": 0.0, "Fwd Packet Length Mean": 8.666667, "Fwd Packet Length Std": 10.263203, "Bwd Packet Length Max": 7300.0, "Bwd Packet Length Min": 0.0, "Bwd Packet Length Mean": 2320.2, "Bwd Packet Length Std": 3022.508, "Flow IAT Mean": 74166.57, "Flow IAT Std": 189753.22, "Flow IAT Max": 504336.0, "Flow IAT Min": 5.0, "Fwd IAT Total": 13978.0, "Fwd IAT Mean": 6989.0, "Fwd IAT Std": 9876.867, "Fwd IAT Max": 13973.0, "Fwd IAT Min": 5.0, "Bwd IAT Total": 518879.0, "Bwd IAT Mean": 129719.75, "Bwd IAT Std": 249826.86, "Bwd IAT Max": 504336.0, "Bwd IAT Min": 169.0, "Fwd PSH Flags": 0.0, "Bwd PSH Flags": 0.0, "Fwd URG Flags": 0.0, "Bwd URG Flags": 0.0, "Fwd Header Length34": 72.0, "Bwd Header Length": 112.0, "Fwd Packets/s": 5.7784986, "Bwd Packets/s": 9.6

In [14]:
print(feature_cols[:10])

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']
